# ParkVision AI — train on PKLot

Slot-level parking occupancy for the ParkVision dashboard.

**Input:** one surveillance frame of a parking lot.
**Output:** every bay labelled occupied or empty, then a lot-level decision.

This notebook downloads the public PKLot v2 640 export (Roboflow, YOLOv8 labels), resplits it 70 / 15 / 15, and fine-tunes YOLOv8n. The Roboflow README that came with the brief describes this export: 12,416 images, auto-oriented, stretched to 640×640, no augmentation applied at export time. Augmentation is applied here during training.

Cite the source if you write the work up:

> Almeida, P., Oliveira, L. S., Silva Jr, E., Britto Jr, A., Koerich, A. PKLot – A robust dataset for parking lot classification. *Expert Systems with Applications*, 42(11):4937–4949, 2015.

Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**, then run the cells in order.


## 1. Install

`ultralytics` trains YOLOv8. `roboflow` downloads the public PKLot project `brad-dwyer/pklot-1tros`, version 2.


In [ ]:
!pip install -q ultralytics roboflow


## 2. Settings

`SUBSET_IMAGES = 1000` keeps the run inside a free Colab session and still clears the brief. Each frame contains many occupied and empty bays, so both classes have far more than 100 instances. Set it to `None` to train on all 12,416 images.

Epochs stay at 20, inside the 15–30 range in the brief.


In [ ]:
from pathlib import Path

SUBSET_IMAGES = 1000  # None = all 12,416 images
EPOCHS = 20
IMAGE_SIZE = 640
SEED = 42
WORK = Path("/content/parkvision")
RAW = WORK / "raw"
READY = WORK / "data"


## 3. Download PKLot

Create a free key at [app.roboflow.com/settings/api](https://app.roboflow.com/settings/api).

In Colab you can store it as a secret named `ROBOFLOW_API_KEY` (the key icon in the left sidebar). Otherwise the next cell asks you to paste it. The key is not written into this notebook.


In [ ]:
import os

api_key = os.environ.get("ROBOFLOW_API_KEY", "").strip()
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("ROBOFLOW_API_KEY").strip()
    except Exception:
        api_key = ""

if not api_key:
    from getpass import getpass
    api_key = getpass("Roboflow API key: ").strip()

if not api_key:
    raise RuntimeError("A Roboflow API key is required to download PKLot.")

import shutil
if RAW.exists():
    shutil.rmtree(RAW)
RAW.mkdir(parents=True, exist_ok=True)

from roboflow import Roboflow

rf = Roboflow(api_key=api_key)
project = rf.workspace("brad-dwyer").project("pklot-1tros")
dataset = project.version(2).download("yolov8", location=str(RAW))
print("Downloaded to", dataset.location)


## 4. Inspect labels

Confirm the two classes, then count how many boxes of each class the export contains. PKLot names are typically `empty` and `occupied`.


In [ ]:
import yaml

downloaded = Path(dataset.location)
yaml_files = list(downloaded.rglob("data.yaml"))
if not yaml_files:
    raise FileNotFoundError(f"No data.yaml under {downloaded}")
source_yaml = yaml_files[0]
source_root = source_yaml.parent
cfg = yaml.safe_load(source_yaml.read_text())
names = cfg["names"]
if isinstance(names, dict):
    class_names = [names[k] for k in sorted(names, key=lambda i: int(i))]
else:
    class_names = list(names)
print("Classes:", class_names)
print("Export root:", source_root)

def label_dirs(root: Path):
    found = []
    for split in ("train", "valid", "val", "test"):
        labels = root / split / "labels"
        images = root / split / "images"
        if labels.is_dir() and images.is_dir():
            found.append((split, images, labels))
    return found

splits = label_dirs(source_root)
print("Splits on disk:", [name for name, _, _ in splits])

from collections import Counter
box_counts = Counter()
image_count = 0
for _, _, labels in splits:
    for label_path in labels.glob("*.txt"):
        image_count += 1
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if parts:
                box_counts[class_names[int(parts[0])]] += 1
print("Images:", image_count)
print("Boxes:", dict(box_counts))


## 5. Subset and resplit

The Roboflow export is already 70 / 20 / 10. The brief asks for **70% train, 15% validation, 15% test**, so the images are pooled and split again with a fixed seed. Resize is already done (640×640 stretch). This step only reorganises files.


In [ ]:
import random
import shutil

pairs = []
for _, images, labels in splits:
    for image_path in images.iterdir():
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        label_path = labels / f"{image_path.stem}.txt"
        if label_path.is_file() and label_path.stat().st_size > 0:
            pairs.append((image_path, label_path))

random.Random(SEED).shuffle(pairs)
if SUBSET_IMAGES is not None:
    pairs = pairs[:SUBSET_IMAGES]
print(f"Using {len(pairs)} images")

n = len(pairs)
n_train = int(n * 0.70)
n_val = int(n * 0.15)
buckets = {
    "train": pairs[:n_train],
    "valid": pairs[n_train:n_train + n_val],
    "test": pairs[n_train + n_val:],
}
for name, items in buckets.items():
    print(f"  {name:5} {len(items):5}  ({len(items) / n:.0%})")

if READY.exists():
    shutil.rmtree(READY)
for name, items in buckets.items():
    img_dir = READY / name / "images"
    lbl_dir = READY / name / "labels"
    img_dir.mkdir(parents=True)
    lbl_dir.mkdir(parents=True)
    for index, (image_path, label_path) in enumerate(items):
        stem = f"{name}-{index:05d}"
        shutil.copy2(image_path, img_dir / f"{stem}{image_path.suffix.lower()}")
        shutil.copy2(label_path, lbl_dir / f"{stem}.txt")

data_yaml = READY / "data.yaml"
data_yaml.write_text(
    yaml.safe_dump(
        {
            "path": str(READY),
            "train": "train/images",
            "val": "valid/images",
            "test": "test/images",
            "nc": len(class_names),
            "names": class_names,
        },
        sort_keys=False,
    )
)
print(data_yaml.read_text())


## 6. Train

YOLOv8n starts from COCO weights and learns the two parking classes. The export had no augmentation, so training adds the variation the brief asks for:

- rotation (`degrees=8`)
- horizontal flip (`fliplr=0.5`)
- brightness (`hsv_v=0.4`)

Vertical flip stays off. These cameras are mounted upright.


In [ ]:
import torch
from ultralytics import YOLO

use_gpu = torch.cuda.is_available()
batch = 16 if use_gpu else 8
print("GPU:" , torch.cuda.get_device_name(0) if use_gpu else "none — training on CPU will be slow")

model = YOLO("yolov8n.pt")
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=batch,
    device=0 if use_gpu else "cpu",
    seed=SEED,
    patience=8,
    degrees=8.0,
    fliplr=0.5,
    flipud=0.0,
    hsv_v=0.4,
    hsv_s=0.3,
    project=str(WORK / "runs"),
    name="parkvision",
    exist_ok=True,
    plots=True,
)


## 7. Evaluate on the test split

This is the 15% the model did not train on. Ultralytics writes a confusion matrix, precision, recall, and mAP next to the run.


In [ ]:
from IPython.display import Image, display

best = WORK / "runs" / "parkvision" / "weights" / "best.pt"
model = YOLO(str(best))
metrics = model.val(
    data=str(data_yaml),
    split="test",
    imgsz=IMAGE_SIZE,
    plots=True,
    project=str(WORK / "runs"),
    name="parkvision-test",
    exist_ok=True,
)

print(f"Precision  {metrics.box.mp:.3f}")
print(f"Recall     {metrics.box.mr:.3f}")
print(f"mAP50      {metrics.box.map50:.3f}")
print(f"mAP50-95   {metrics.box.map:.3f}")

for picture in (
    WORK / "runs" / "parkvision" / "results.png",
    WORK / "runs" / "parkvision" / "confusion_matrix.png",
    WORK / "runs" / "parkvision-test" / "confusion_matrix.png",
):
    if picture.is_file():
        print(picture.name)
        display(Image(filename=str(picture)))


## 8. Download the weights

Save the file as `models/best.pt` in the ParkVision project, then restart the Streamlit app. Empty bays draw in green, occupied bays in red, and the dashboard applies the 40% / 75% congestion rules.


In [ ]:
import shutil
from google.colab import files

export_path = Path("/content/best.pt")
shutil.copy2(best, export_path)
print("Weights:", export_path, f"({export_path.stat().st_size / 1e6:.1f} MB)")
files.download(str(export_path))
